In [ ]:
import numpy as np
import pandas as pd
from skimage.draw    import polygon
from sklearn.cluster         import KMeans
from sklearn.preprocessing   import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import matplotlib.patches    as mpatches
import matplotlib.colors     as mcolors
from skimage.draw    import polygon
import pandas as pd
from sklearn.metrics         import classification_report, confusion_matrix, f1_score
import matplotlib.colors     as mcolors

WINDOW, HALF = 25, 24
TRAIN_DAYS = (
    list(range(1230,1300)) 
)
VAL_DAYS = (
    list(range(2557, 2622)) 
)
RNG        = 42
KM         = 10
EPOCHS     = 10
BATCH      = 256
META_FILE  = "patches_data.npz"
IMG_FILE   = "patches_imgs.h5"
_SPLIT_KEYS = ["X_basic", "y_all", "coords", "types_arr", "regions_codes", "days_arr"]
REGIONS9   = ["NW","N","NE","W","C","E","SW","S","SE"]
ROI_TYPES  = ["COL","CL","COH","NROI"]
TYPE_COLORS = {"COL":"red","CL":"blue","COH":"green","NROI":"orange"}

def create_roi_mask(df, day, shape):
    mask = np.zeros(shape, dtype=np.int8)
    day_data = df[df["Day"]==day]
    for name, grp in day_data.groupby("Name"):
        pts = grp[["x","y"]].values
        if len(pts)<3:
            continue
        xs, ys = pts[:,0].astype(np.int32), pts[:,1].astype(np.int32)
        rows = (shape[0]-1) - ys
        cols = xs
        rr, cc = polygon(rows, cols, shape=shape)
        code = ROI_TYPES.index(grp["Label"].iat[0]) + 1
        mask[rr,cc] = code
    return mask

def load_grids(day):
    P = pd.read_csv(f"pressure/day{day}.txt", sep=r"\s+", header=None).values.astype(np.float32)
    W = pd.read_csv(f"wind/day{day}.txt",    sep=r"\s+", header=None).values.astype(np.float32)
    return P, W

type_to_label = {"COL": 0, "CL": 1, "COH": 2, "NROI": 3, "BG": 4}
label_to_type = {v: k for k, v in type_to_label.items()}
b_type_to_label = {"NROI":0,"ROI": 1}

cmap_types = mcolors.ListedColormap(["lightgrey"] + [TYPE_COLORS[t] for t in ROI_TYPES])
type_code  = {t:i+1 for i,t in enumerate(ROI_TYPES)}

roi_df = pd.read_csv("roi_data_with_status.csv")
required = {"Day","Name","x","y","Label"}
if not required.issubset(roi_df.columns):
    raise RuntimeError(f"roi_data_with_status.csv must contain columns: {required}")

raw = np.load(META_FILE, allow_pickle=True)
tr_data = {k: raw[f"tr_{k}"] for k in _SPLIT_KEYS}
te_data = {k: raw[f"te_{k}"] for k in _SPLIT_KEYS}

X_basic     = np.vstack([tr_data['X_basic'], te_data['X_basic']])
y_all       = np.concatenate([tr_data['y_all'], te_data['y_all']])
coords      = np.vstack([tr_data['coords'], te_data['coords']])
types_arr   = np.concatenate([tr_data['types_arr'], te_data['types_arr']])
regions_codes = np.concatenate([tr_data['regions_codes'], te_data['regions_codes']])
days_arr    = np.concatenate([tr_data['days_arr'], te_data['days_arr']])

# Convert region codes to one-hot
reg_ohe = np.zeros((len(regions_codes), 9), dtype=np.float32)
reg_ohe[np.arange(len(regions_codes)), regions_codes] = 1

# Rebuild regions array for compatibility
regions = np.array([REGIONS9[c] for c in regions_codes])

# === K-MEANS ===
train_mask = np.isin(days_arr, TRAIN_DAYS)
val_mask = np.isin(days_arr, VAL_DAYS)

print("Running K-means clustering...")
# km = KMeans(n_clusters=KM, random_state=RNG, n_init=10).fit(X_basic)
# clus_ohe = np.zeros((len(X_basic), KM), dtype=np.float32)
# clus_ohe[np.arange(len(X_basic)), km.labels_] = 1
# dists = np.linalg.norm(X_basic - km.cluster_centers_[km.labels_], axis=1).reshape(-1, 1)

# X_feat = np.hstack([X_basic, reg_ohe, clus_ohe, dists])

X_feat = np.load('X_feat.npz')['x']
scaler = StandardScaler()
X_feat_tr  = scaler.fit_transform(X_feat[train_mask])
X_feat_val = scaler.transform(X_feat[val_mask])

y_tr, y_val   = y_all[train_mask], y_all[val_mask]
coords_tr, coords_val  = coords[train_mask], coords[val_mask]
types_tr, types_val  = types_arr[train_mask], types_arr[val_mask]
days_tr, days_val  = days_arr[train_mask], days_arr[val_mask]

b_y_tr = np.array([0 if t in (3, 4) else 1 for t in y_tr])
b_y_val = np.array([0 if t in (3, 4) else 1 for t in y_val])

print("Train class counts:", np.bincount(b_y_tr))
print("Val class counts:",   np.bincount(b_y_val))

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(b_y_tr),
    y=b_y_tr
)

class_weights = torch.tensor(weights, dtype=torch.float32).to(device)  

class BDNN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128), nn.ReLU(),
            nn.Linear(128, 64),         nn.ReLU(),
            nn.Linear(64, 32),          nn.ReLU(),
            nn.Linear(32, 2),
        )
    def forward(self, x):
        return self.net(x)

bdnn = BDNN(X_feat_tr.shape[1]).to(device)  

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(bdnn.parameters(), lr=1e-4)

def make_loader(X, y, batch_size, shuffle):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_feat_tr, b_y_tr, BATCH, shuffle=True)
val_loader   = make_loader(X_feat_te, b_y_te, BATCH, shuffle=False)

for epoch in range(1, EPOCHS + 1):
    bdnn.train()
    train_loss = train_correct = train_total = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)  # ← GPU

        optimizer.zero_grad()
        logits = bdnn(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item() * len(y_batch)
        train_correct += (logits.argmax(1) == y_batch).sum().item()
        train_total   += len(y_batch)

    bdnn.eval()
    val_loss = val_correct = val_total = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)  

            logits      = bdnn(X_batch)
            loss        = criterion(logits, y_batch)
            val_loss    += loss.item() * len(y_batch)
            val_correct += (logits.argmax(1) == y_batch).sum().item()
            val_total   += len(y_batch)

    train_loss /= train_total;  train_acc = train_correct / train_total
    val_loss   /= val_total;    val_acc   = val_correct   / val_total

    print(f"Epoch {epoch}/{EPOCHS}  "
          f"loss: {train_loss:.4f}  acc: {train_acc:.4f}  "
          f"val_loss: {val_loss:.4f}  val_acc: {val_acc:.4f}")

torch.save(bdnn.state_dict(), "Stage1DNN.pt")
print('Saved DNN')

In [ ]:
bdnn = BDNN(X_feat_val.shape[1]).to(device)
bdnn.load_state_dict(torch.load("Stage1DNN.pt"))
bdnn.eval()

with torch.no_grad():
    X_val_tensor = torch.tensor(X_feat_val, dtype=torch.float32).to(device)
    probs_val    = torch.softmax(bdnn(X_val_tensor), dim=1)[:, 1].cpu().numpy()
    y_type_pred_val = (probs_val > 0.5).astype(int)
    roi_mask_val    = probs_val > 0.5
    X_stage2_val    = X_feat_val[roi_mask_val]
    y_stage2_val    = y_val[roi_mask_val]

b_label_to_type = {0: "NROI",1: "ROI"}
pred_types_strings = [b_label_to_type[int(p)] for p in y_type_pred_val]
true_types_strings = [b_label_to_type[int(t)] for t in b_y_tr]
binary_labels = ["ROI", "NROI"]

cm_types_2 = np.array(confusion_matrix(
    true_types_strings,
    pred_types_strings,
    labels=binary_labels
))

plt.figure(figsize=(5, 4))
plt.imshow(cm_types_2, cmap="Blues")
plt.title("ROI/NROI Confusion Matrix (Train)")
plt.xticks(range(len(binary_labels)), binary_labels)
plt.yticks(range(len(binary_labels)), binary_labels)
plt.xlabel("Predicted Type")
plt.ylabel("True Type")
for i in range(len(binary_labels)):
    for j in range(len(binary_labels)):
        val = cm_types_2[i, j]
        plt.text(j, i, val, ha="center", va="center",
                color="white" if val > cm_types_2.max()/2 else "black")
plt.tight_layout()
plt.show()

print("\n=== Train Binary Classification Report ===")
print(classification_report(true_types_strings, pred_types_strings, labels=binary_labels))

In [ ]:
X_stage2_tr = X_stage2_val[valid_mask_tr]
y_stage2_tr = y_stage2_val[valid_mask_tr]

X_stage2_te = X_stage2_val[valid_mask_te]
y_stage2_te = y_stage2_val[valid_mask_te]

print(np.unique(y_stage2_tr, return_counts=True))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_stage2_tr),
    y=y_stage2_tr
)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)  

class CDNN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),         nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32),          nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 3),
        )
    def forward(self, x):
        return self.net(x)

cdnn = CDNN(X_stage2_tr.shape[1]).to(device)  

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(cdnn.parameters(), lr=1e-4)

def make_loader(X, y, batch_size, shuffle):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_stage2_tr, y_stage2_tr, BATCH, shuffle=True)
val_loader   = make_loader(X_stage2_te, y_stage2_te, BATCH, shuffle=False)

for epoch in range(1, EPOCHS + 1):
    cdnn.train()
    train_loss = train_correct = train_total = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)  # ← GPU

        optimizer.zero_grad()
        logits = cdnn(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item() * len(y_batch)
        train_correct += (logits.argmax(1) == y_batch).sum().item()
        train_total   += len(y_batch)

    cdnn.eval()
    val_loss = val_correct = val_total = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)  

            logits      = cdnn(X_batch)
            loss        = criterion(logits, y_batch)
            val_loss    += loss.item() * len(y_batch)
            val_correct += (logits.argmax(1) == y_batch).sum().item()
            val_total   += len(y_batch)

    train_loss /= train_total;  train_acc = train_correct / train_total
    val_loss   /= val_total;    val_acc   = val_correct   / val_total

    print(f"Epoch {epoch}/{EPOCHS}  "
          f"loss: {train_loss:.4f}  acc: {train_acc:.4f}  "
          f"val_loss: {val_loss:.4f}  val_acc: {val_acc:.4f}")

torch.save(cdnn.state_dict(), "Stage2DNN.pt")
print('Saved DNN')
valid_mask_tr = (y_stage2_tr != 3) & (y_stage2_tr != 4)
X_stage2_tr = X_stage2_tr[valid_mask_tr]
y_stage2_tr = y_stage2_tr[valid_mask_tr]

valid_mask_te = (y_stage2_te != 3) & (y_stage2_te != 4)
X_stage2_te = X_stage2_te[valid_mask_te]
y_stage2_te = y_stage2_te[valid_mask_te]

print(np.unique(y_stage2_tr, return_counts=True))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_stage2_tr),
    y=y_stage2_tr
)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)  

class CDNN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),         nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32),          nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 3),
        )
    def forward(self, x):
        return self.net(x)

cdnn = CDNN(X_stage2_tr.shape[1]).to(device)  

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(cdnn.parameters(), lr=1e-4)

def make_loader(X, y, batch_size, shuffle):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_stage2_tr, y_stage2_tr, BATCH, shuffle=True)
val_loader   = make_loader(X_stage2_te, y_stage2_te, BATCH, shuffle=False)

for epoch in range(1, EPOCHS + 1):
    cdnn.train()
    train_loss = train_correct = train_total = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)  # ← GPU

        optimizer.zero_grad()
        logits = cdnn(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item() * len(y_batch)
        train_correct += (logits.argmax(1) == y_batch).sum().item()
        train_total   += len(y_batch)

    cdnn.eval()
    val_loss = val_correct = val_total = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)  

            logits      = cdnn(X_batch)
            loss        = criterion(logits, y_batch)
            val_loss    += loss.item() * len(y_batch)
            val_correct += (logits.argmax(1) == y_batch).sum().item()
            val_total   += len(y_batch)

    train_loss /= train_total;  train_acc = train_correct / train_total
    val_loss   /= val_total;    val_acc   = val_correct   / val_total

    print(f"Epoch {epoch}/{EPOCHS}  "
          f"loss: {train_loss:.4f}  acc: {train_acc:.4f}  "
          f"val_loss: {val_loss:.4f}  val_acc: {val_acc:.4f}")

torch.save(cdnn.state_dict(), "Stage2DNN.pt")
print('Saved DNN')